# Phase 2: 予測モデル（v2・修正版）

`docs/requirements.md` 5章に基づく。LightGBMをベースラインとし、特徴量の質で戦う。モデルの複雑化はしない。

## v1 からの変更点（Phase 2 検証を受けた修正）

1. **PR-AUCを離脱=正例で再計算**: v1は完走（多数派、約94%）を正例としており、ベースラインPR-AUCが0.90前後と
   最初から高すぎて識別力が見えなかった。少数派である**離脱を正例**として全指標を再計算する。
2. **埋め込みの新作パス検証**: `anime_avg_completion_rate` を除外した特徴量セットで埋め込みあり/なしを比較する。
   これは2021年以降の新作（過去実績なし）の判定精度に直結する。
3. **人気順トップ20との重複比較を修正**: v1は「ユーザーが既に見た作品」内での並べ替えになっており、
   UI仕様書4.8（未視聴作品の推薦画面）と対応していなかった。**未回答の作品200本**を候補として抽出し直す。

## 離脱の定義（変更）

**定義A（離脱=4のみ）を主定義とする。** 記録習慣の検証（v1ノートブック参照）で、
定義Bでプロファイルを構築するとむしろ定義Aラベルの予測精度が下がる（AUC 0.838→0.831）ことが
確認されたため、事前に合意した判定ルールに従い定義Aへ戻す。

- 学習ラベル: 定義A（離脱=4のみ）
- プロファイル入力: 定義A（保留は使わない）
- UI: 3ボタンのまま変更しない。ユーザーの「途中で止まった」をMALの離脱(4)相当の意思表示として扱う。
  保留(3)は記録習慣の差（相関-0.77）を含むため学習から除外する
- **`data/dropout_curves.json` / `data/question_pool.json` は定義Bのまま維持する**（脱落曲線は
  「どこで人が止まるか」の集計であり、ラベル定義とは目的が異なるため再生成不要）


In [1]:
import re
import sys
import time
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score
from sentence_transformers import SentenceTransformer

sys.path.insert(0, "..")
from api.services.profile import Response, compute_endurance_episodes, compute_completion_rate

RAW_DIR = "../data/raw"
PROC_DIR = "../data/processed"
ANIMELIST_PATH = f"{RAW_DIR}/animelist.csv"
ANIME_PATH = f"{RAW_DIR}/anime.csv"
SYNOPSIS_PATH = f"{RAW_DIR}/anime_with_synopsis.csv"

SEED = 42
N_USERS_TARGET = 15_000
CHUNKSIZE = 15_000_000
MAX_N = 30
N_TIERS = [3, 5, 10, 20, 30]
TARGET_CAP_PER_USER = 40

t_total0 = time.time()


## 1. データ分割（user_id単位、70/15/15）

In [2]:
unique_chunks = []
for chunk in pd.read_csv(ANIMELIST_PATH, usecols=["user_id"], dtype={"user_id": "int32"}, chunksize=CHUNKSIZE):
    unique_chunks.append(chunk["user_id"].unique())
unique_users = np.unique(np.concatenate(unique_chunks))

rng = np.random.default_rng(SEED)
sampled_users = rng.choice(unique_users, size=min(N_USERS_TARGET, len(unique_users)), replace=False)
sampled_users_set = set(sampled_users.tolist())

perm = rng.permutation(sampled_users)
n = len(perm)
n_train, n_val = int(n * 0.70), int(n * 0.15)
train_users = perm[:n_train]
val_users = perm[n_train:n_train + n_val]
test_users = perm[n_train + n_val:]
print(f"users: {n:,} | train={len(train_users):,} val={len(val_users):,} test={len(test_users):,}")


users: 15,000 | train=10,500 val=2,250 test=2,250


In [3]:
dtypes = {"user_id": "int32", "anime_id": "int32", "watching_status": "int8"}
parts = []
for chunk in pd.read_csv(ANIMELIST_PATH, dtype=dtypes, usecols=list(dtypes.keys()), chunksize=CHUNKSIZE):
    mask = chunk["user_id"].isin(sampled_users_set) & chunk["watching_status"].isin([2, 3, 4])
    if mask.any():
        parts.append(chunk.loc[mask])
labels_all = pd.concat(parts, ignore_index=True)

# 定義A: on-hold(3)は入力候補・ターゲットいずれからも除外する
labels = labels_all[labels_all["watching_status"] != 3]
print(f"labels(定義A) rows: {len(labels):,} | users covered: {labels['user_id'].nunique():,}")


labels(定義A) rows: 3,340,506 | users covered: 14,788


## 2. 作品側の静的特徴量とtrainのみからの平均完走率（定義A）

In [4]:
anime = pd.read_csv(ANIME_PATH)

def extract_year(aired):
    m = re.search(r"(19|20)\d{2}", str(aired))
    return int(m.group()) if m else None

anime["year"] = anime["Aired"].apply(extract_year)
anime["episodes_num"] = pd.to_numeric(anime["Episodes"], errors="coerce")
anime["score_num"] = pd.to_numeric(anime["Score"], errors="coerce")
anime["members_num"] = pd.to_numeric(anime["Members"], errors="coerce")
anime["members_log"] = np.log1p(anime["members_num"])
anime["genre_list"] = anime["Genres"].apply(lambda s: [g.strip() for g in str(s).split(",")] if pd.notna(s) else [])
anime["source"] = anime["Source"].fillna("Unknown")

all_genres = sorted({g for gl in anime["genre_list"] for g in gl if g and g != "Unknown"})
genre_cols = {f"genre_{g}": anime["genre_list"].apply(lambda gl: 1 if g in gl else 0) for g in all_genres}
genre_df = pd.DataFrame(genre_cols)

anime_feat = pd.concat([
    anime[["MAL_ID", "year", "episodes_num", "score_num", "members_num", "members_log", "source", "genre_list"]],
    genre_df,
], axis=1).rename(columns={"MAL_ID": "anime_id"}).set_index("anime_id")
anime_feat["popularity_percentile"] = anime_feat["members_num"].rank(pct=True)
print(f"unique genres: {len(all_genres)} | anime_feat shape: {anime_feat.shape}")


unique genres: 43 | anime_feat shape: (17562, 51)


In [5]:
train_user_set = set(train_users.tolist())
train_labels = labels[labels["user_id"].isin(train_user_set)]
n_completed = train_labels[train_labels["watching_status"] == 2].groupby("anime_id").size()
n_dropped_A = train_labels[train_labels["watching_status"] == 4].groupby("anime_id").size()
anime_avg = pd.DataFrame({"n_completed": n_completed, "n_dropped_A": n_dropped_A}).fillna(0)
anime_avg["anime_avg_completion_rate_A"] = anime_avg["n_completed"] / (anime_avg["n_completed"] + anime_avg["n_dropped_A"])
print(f"train-onlyの平均完走率(定義A)を持つ作品数: {len(anime_avg):,}")


train-onlyの平均完走率(定義A)を持つ作品数: 15,132


## 3. あらすじ埋め込み（multilingual-e5-small）

`data/processed/` にキャッシュ済みなら再利用し、無ければ計算する。

In [6]:
import os
if os.path.exists(f"{PROC_DIR}/anime_embeddings.npy"):
    embeddings = np.load(f"{PROC_DIR}/anime_embeddings.npy")
    emb_ids = np.load(f"{PROC_DIR}/anime_embeddings_ids.npy")
    print(f"cached embeddings loaded: {embeddings.shape}")
else:
    syn = pd.read_csv(SYNOPSIS_PATH)
    syn = syn.dropna(subset=["sypnopsis"])
    syn = syn[syn["sypnopsis"].str.strip().str.len() > 0]
    syn = syn[syn["sypnopsis"] != "No synopsis information has been added to this title."]
    texts = ("query: " + syn["sypnopsis"].astype(str).str.slice(0, 2000)).tolist()
    emb_ids = syn["MAL_ID"].values
    model_e5 = SentenceTransformer("intfloat/multilingual-e5-small")
    embeddings = model_e5.encode(texts, batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    os.makedirs(PROC_DIR, exist_ok=True)
    np.save(f"{PROC_DIR}/anime_embeddings.npy", embeddings.astype(np.float32))
    np.save(f"{PROC_DIR}/anime_embeddings_ids.npy", emb_ids)
    print(f"computed & cached embeddings: {embeddings.shape}")

emb_lookup = {int(aid): embeddings[i] for i, aid in enumerate(emb_ids)}


cached embeddings loaded: (16206, 384)


## 4. question_pool・ルックアップの準備

`question_pool.json` は Phase 1（定義B）のまま。ここでは入力候補ソースとして使うだけであり、
実際にどの回答を入力に使うかは定義A（status 2,4のみ）に絞り込む。

In [7]:
pool = json.load(open("../data/question_pool.json"))
pool_ids = set(p["anime_id"] for p in pool)

genres_lookup = anime_feat["genre_list"].to_dict()
episodes_lookup = anime_feat["episodes_num"].to_dict()
members_lookup = anime_feat["members_num"].fillna(0).to_dict()
popularity_lookup = anime_feat["popularity_percentile"].to_dict()

GENRE_COLS = [f"genre_{g}" for g in all_genres]
STATIC_COLS = ["year", "episodes_num", "score_num", "members_log", "source", "popularity_percentile"] + GENRE_COLS
labels_by_user = {uid: g for uid, g in labels.groupby("user_id")}
print("pool_ids:", len(pool_ids))


pool_ids: 250


## 5. 入力制限シミュレーション: 行データセットの構築（定義A）

正例（`label_dropped`）は離脱(4)。完走(2)が負例。on-holdは母集団の時点で除外済み。

In [8]:
def cos_sim(a, b):
    if a is None or b is None:
        return np.nan
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0:
        return np.nan
    return float(np.dot(a, b) / (na * nb))

def make_response(anime_id, status):
    return Response(
        anime_id=int(anime_id), label=("completed" if status == 2 else "dropped"),
        episodes=int(episodes_lookup.get(anime_id) or 0),
        genres=genres_lookup.get(anime_id) or [],
        members=int(members_lookup.get(anime_id) or 0),
    )

def profile_vectors(responses):
    comp_embs = [emb_lookup[r.anime_id] for r in responses if r.label == "completed" and r.anime_id in emb_lookup]
    drop_embs = [emb_lookup[r.anime_id] for r in responses if r.label == "dropped" and r.anime_id in emb_lookup]
    comp_vec = np.mean(comp_embs, axis=0) if comp_embs else None
    drop_vec = np.mean(drop_embs, axis=0) if drop_embs else None
    return comp_vec, drop_vec

def build_target_rows(user_group, rng):
    rows = user_group[["anime_id", "watching_status"]].values.tolist()
    pool_rows = [r for r in rows if r[0] in pool_ids]
    nonpool_rows = [r for r in rows if r[0] not in pool_ids]
    perm_idx = rng.permutation(len(pool_rows))
    pool_rows_shuffled = [pool_rows[i] for i in perm_idx]
    input_pool_rows = pool_rows_shuffled[:MAX_N]
    holdout_pool_rows = pool_rows_shuffled[MAX_N:]
    target_rows = holdout_pool_rows + nonpool_rows
    if len(target_rows) > TARGET_CAP_PER_USER:
        idx = rng.choice(len(target_rows), size=TARGET_CAP_PER_USER, replace=False)
        target_rows = [target_rows[i] for i in idx]
    return input_pool_rows, target_rows

def build_dataset_for_split(user_ids, split_name):
    all_rows = []
    n_used = 0
    for uid in user_ids:
        if uid not in labels_by_user:
            continue
        g = labels_by_user[uid]
        rng = np.random.default_rng(SEED + int(uid))
        input_pool_rows, target_rows = build_target_rows(g, rng)
        if len(target_rows) == 0 or len(input_pool_rows) == 0:
            continue
        n_used += 1
        for n in N_TIERS:
            responses = [make_response(aid, st) for aid, st in input_pool_rows[:n]]
            endurance = compute_endurance_episodes(responses)
            completion_rate = compute_completion_rate(responses)
            comp_vec, drop_vec = profile_vectors(responses)
            comp_pop = [popularity_lookup.get(r.anime_id, np.nan) for r in responses if r.label == "completed"]
            user_mean_pop = float(np.mean(comp_pop)) if comp_pop else np.nan
            genre_rate = {}
            for g_name in all_genres:
                c = sum(1 for r in responses if r.label == "completed" and g_name in r.genres)
                d = sum(1 for r in responses if r.label == "dropped" and g_name in r.genres)
                if c + d > 0:
                    genre_rate[g_name] = c / (c + d)
            for aid, status in target_rows:
                a_genres = genres_lookup.get(aid) or []
                a_episodes = episodes_lookup.get(aid)
                a_pop = popularity_lookup.get(aid, np.nan)
                a_emb = emb_lookup.get(aid)
                cos_c = cos_sim(a_emb, comp_vec)
                cos_d = cos_sim(a_emb, drop_vec)
                ep_gap = (a_episodes - endurance) if (a_episodes is not None and endurance is not None) else np.nan
                gmatch = [genre_rate[g_name] for g_name in a_genres if g_name in genre_rate]
                genre_match_rate = float(np.mean(gmatch)) if gmatch else np.nan
                pop_div = (a_pop - user_mean_pop) if not np.isnan(user_mean_pop) else np.nan
                all_rows.append({
                    "split": split_name, "N": n, "user_id": uid, "anime_id": aid, "n_input": len(responses),
                    "endurance_episodes": endurance, "user_completion_rate": completion_rate,
                    "cos_completed": cos_c, "cos_dropped": cos_d, "episode_gap": ep_gap,
                    "genre_match_completion_rate": genre_match_rate, "popularity_divergence": pop_div,
                    "label_dropped": 1 if status == 4 else 0,
                })
    return pd.DataFrame(all_rows), n_used

def attach_static_features(df):
    static = anime_feat[STATIC_COLS].reset_index()
    df = df.merge(static, on="anime_id", how="left")
    avg = anime_avg.reset_index().rename(columns={"index": "anime_id"})[["anime_id", "anime_avg_completion_rate_A"]]
    df = df.merge(avg, on="anime_id", how="left")
    df["anime_avg_completion_rate_A"] = df["anime_avg_completion_rate_A"].fillna(df["anime_avg_completion_rate_A"].mean())
    df["source"] = df["source"].astype("category")
    return df


In [9]:
t3 = time.time()
datasets = {}
for split_name, user_ids in [("train", train_users), ("val", val_users), ("test", test_users)]:
    df, n_used = build_dataset_for_split(user_ids, split_name)
    df = attach_static_features(df)
    datasets[split_name] = df
    print(f"[{split_name}] users_used={n_used:,} rows={len(df):,} pos_rate(N=20)={df[df.N==20]['label_dropped'].mean():.4f}")
print(f"elapsed: {time.time()-t3:.1f}s")

train_df, val_df, test_df = datasets["train"], datasets["val"], datasets["test"]


[train] users_used=10,004 rows=1,805,260 pos_rate(N=20)=0.0564
[val] users_used=2,139 rows=390,660 pos_rate(N=20)=0.0557
[test] users_used=2,146 rows=385,285 pos_rate(N=20)=0.0556
elapsed: 45.5s


## 6. 評価関数とLightGBM学習ヘルパー

In [10]:
def auc_pr(y_true, y_score):
    y_true = np.asarray(y_true); y_score = np.asarray(y_score)
    mask = ~pd.isna(y_score) & ~pd.isna(y_true)
    y_true, y_score = y_true[mask], y_score[mask]
    if len(np.unique(y_true)) < 2:
        return np.nan, np.nan
    return roc_auc_score(y_true, y_score), average_precision_score(y_true, y_score)

def train_lgbm(feat_cols, tr, va, cat_cols=None):
    cat_cols = [c for c in (cat_cols or []) if c in feat_cols]
    dtrain = lgb.Dataset(tr[feat_cols], label=tr["label_dropped"], categorical_feature=cat_cols or "auto", free_raw_data=False)
    dval = lgb.Dataset(va[feat_cols], label=va["label_dropped"], reference=dtrain, categorical_feature=cat_cols or "auto", free_raw_data=False)
    params = {"objective": "binary", "metric": "auc", "verbosity": -1, "learning_rate": 0.05,
              "num_leaves": 31, "min_data_in_leaf": 50, "feature_fraction": 0.8, "bagging_fraction": 0.8,
              "bagging_freq": 1, "seed": 42}
    return lgb.train(params, dtrain, num_boost_round=500, valid_sets=[dval],
                      callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)])

CAT_COLS = ["source"]
USER_COLS = ["endurance_episodes", "user_completion_rate", "genre_match_completion_rate", "episode_gap"]


## 7. 評価表【必須・正例=離脱】（N=20基準）

In [11]:
te20 = test_df[test_df.N == 20].copy()
tr20, va20 = train_df[train_df.N == 20], val_df[val_df.N == 20]
baseline_pos_rate = te20["label_dropped"].mean()
print(f"ベースライン正例率（離脱率）: {baseline_pos_rate:.4f}")

results_main = []
rng2 = np.random.default_rng(0)
auc, pr = auc_pr(te20["label_dropped"], rng2.random(len(te20)))
results_main.append({"stage": "ランダム", "auc": auc, "pr_auc": pr})

te20["pop_score"] = 1 - te20["popularity_percentile"]
auc, pr = auc_pr(te20["label_dropped"], te20["pop_score"])
results_main.append({"stage": "人気順（登録者数）", "auc": auc, "pr_auc": pr})

te20["avg_score"] = 1 - te20["anime_avg_completion_rate_A"]
auc, pr = auc_pr(te20["label_dropped"], te20["avg_score"])
results_main.append({"stage": "作品の平均完走率のみ", "auc": auc, "pr_auc": pr})

feat1 = ["anime_avg_completion_rate_A"] + USER_COLS + STATIC_COLS
m1 = train_lgbm(feat1, tr20, va20, CAT_COLS)
auc, pr = auc_pr(te20["label_dropped"], m1.predict(te20[feat1], num_iteration=m1.best_iteration))
results_main.append({"stage": "+ユーザー特徴（耐久話数・ジャンル）", "auc": auc, "pr_auc": pr})

feat2 = feat1 + ["cos_completed"]
m2 = train_lgbm(feat2, tr20, va20, CAT_COLS)
auc, pr = auc_pr(te20["label_dropped"], m2.predict(te20[feat2], num_iteration=m2.best_iteration))
results_main.append({"stage": "+埋め込み類似度（完走ベクトルのみ）", "auc": auc, "pr_auc": pr})

feat3 = feat1 + ["cos_completed", "cos_dropped"]
model_full = train_lgbm(feat3, tr20, va20, CAT_COLS)
auc, pr = auc_pr(te20["label_dropped"], model_full.predict(te20[feat3], num_iteration=model_full.best_iteration))
results_main.append({"stage": "+離脱ベクトル", "auc": auc, "pr_auc": pr})

pd.DataFrame(results_main)


ベースライン正例率（離脱率）: 0.0556


,stage,auc,pr_auc
0,ランダム,0.496588,0.054936
1,人気順（登録者数）,0.526025,0.059971
2,作品の平均完走率のみ,0.772355,0.222626
3,+ユーザー特徴（耐久話数・ジャンル）,0.837271,0.327601
4,+埋め込み類似度（完走ベクトルのみ）,0.837728,0.323469
5,+離脱ベクトル,0.837859,0.325364


## 8. 追加分析①: 入力本数 × 精度の曲線

In [12]:
n_curve = []
for n in N_TIERS:
    trN, vaN, teN = train_df[train_df.N==n], val_df[val_df.N==n], test_df[test_df.N==n]
    mN = train_lgbm(feat3, trN, vaN, CAT_COLS)
    auc, pr = auc_pr(teN["label_dropped"], mN.predict(teN[feat3], num_iteration=mN.best_iteration))
    n_curve.append({"N": n, "auc": auc, "pr_auc": pr})
n_curve_df = pd.DataFrame(n_curve)
n_curve_df


,N,auc,pr_auc
0,3,0.796754,0.259845
1,5,0.807785,0.280622
2,10,0.827808,0.308644
3,20,0.837859,0.325364
4,30,0.840900,0.331547


## 9. 追加分析②: 入力種別ごとの寄与（定義A: 「途中で止まった」=離脱(4)のみ）

In [13]:
dtypes_r = {"user_id": "int32", "anime_id": "int32", "rating": "int8", "watching_status": "int8"}
parts_r = []
for chunk in pd.read_csv(ANIMELIST_PATH, dtype=dtypes_r, chunksize=CHUNKSIZE):
    mask = chunk["user_id"].isin(sampled_users_set) & (chunk["watching_status"] == 2)
    if mask.any():
        parts_r.append(chunk.loc[mask, ["user_id", "anime_id", "rating"]])
ratings_df = pd.concat(parts_r, ignore_index=True)
rating_lookup = {(u, a): r for u, a, r in zip(ratings_df["user_id"], ratings_df["anime_id"], ratings_df["rating"])}
LOVED_RATING_THRESHOLD = 8

def filter_input_by_type(input_rows_n20, uid, stage):
    out = []
    for aid, status in input_rows_n20:
        if status == 2:
            is_loved = rating_lookup.get((uid, aid), 0) >= LOVED_RATING_THRESHOLD
            if stage == "loved_only" and not is_loved:
                continue
            out.append((aid, status))
        else:
            if stage != "plus_dropped":
                continue
            out.append((aid, status))
    return out

def build_stage_rows(user_ids, stage):
    all_rows = []
    for uid in user_ids:
        if uid not in labels_by_user:
            continue
        g = labels_by_user[uid]
        rng = np.random.default_rng(SEED + int(uid))
        input_pool_rows, target_rows = build_target_rows(g, rng)
        if len(target_rows) == 0 or len(input_pool_rows) == 0:
            continue
        filtered = filter_input_by_type(input_pool_rows[:20], uid, stage)
        responses = [make_response(aid, st) for aid, st in filtered]
        endurance = compute_endurance_episodes(responses)
        completion_rate = compute_completion_rate(responses)
        comp_vec, drop_vec = profile_vectors(responses)
        comp_pop = [popularity_lookup.get(r.anime_id, np.nan) for r in responses if r.label == "completed"]
        user_mean_pop = float(np.mean(comp_pop)) if comp_pop else np.nan
        genre_rate = {}
        for g_name in all_genres:
            c = sum(1 for r in responses if r.label == "completed" and g_name in r.genres)
            d = sum(1 for r in responses if r.label == "dropped" and g_name in r.genres)
            if c + d > 0:
                genre_rate[g_name] = c / (c + d)
        for aid, status in target_rows:
            a_genres = genres_lookup.get(aid) or []
            a_episodes = episodes_lookup.get(aid)
            a_emb = emb_lookup.get(aid)
            cos_c = cos_sim(a_emb, comp_vec); cos_d = cos_sim(a_emb, drop_vec)
            ep_gap = (a_episodes - endurance) if (a_episodes is not None and endurance is not None) else np.nan
            gmatch = [genre_rate[g_name] for g_name in a_genres if g_name in genre_rate]
            genre_match_rate = float(np.mean(gmatch)) if gmatch else np.nan
            a_pop = popularity_lookup.get(aid, np.nan)
            pop_div = (a_pop - user_mean_pop) if not np.isnan(user_mean_pop) else np.nan
            all_rows.append({"user_id": uid, "anime_id": aid, "endurance_episodes": endurance,
                              "user_completion_rate": completion_rate, "cos_completed": cos_c, "cos_dropped": cos_d,
                              "episode_gap": ep_gap, "genre_match_completion_rate": genre_match_rate,
                              "popularity_divergence": pop_div, "label_dropped": 1 if status == 4 else 0})
    return pd.DataFrame(all_rows)

input_type_results = []
for stage, label in [("loved_only", "「好き」のみ"), ("plus_completed", "+「完走」"), ("plus_dropped", "+「途中で止まった」")]:
    tr_s = attach_static_features(build_stage_rows(train_users, stage))
    va_s = attach_static_features(build_stage_rows(val_users, stage))
    te_s = attach_static_features(build_stage_rows(test_users, stage))
    m_s = train_lgbm(feat3, tr_s, va_s, CAT_COLS)
    auc, pr = auc_pr(te_s["label_dropped"], m_s.predict(te_s[feat3], num_iteration=m_s.best_iteration))
    input_type_results.append({"stage": label, "auc": auc, "pr_auc": pr})

input_type_df = pd.DataFrame(input_type_results)
input_type_df


,stage,auc,pr_auc
0,「好き」のみ,0.780275,0.233166
1,+「完走」,0.777542,0.229826
2,+「途中で止まった」,0.837859,0.325364


## 10. 追加分析③: 人気順トップ20 と 個人最適トップ20 の重複件数（再設計版）

**修正点**: v1は「ユーザーが既に見た作品」内での比較になっていた（UI仕様書4.8の意図と不一致）。
ここではテストユーザーごとに**未回答の作品から200本**を候補として抽出し直す
（候補ユニバースは登録者数上位3,000作品。question_pool選定と同じ考え方）。

In [14]:
registrations = labels[labels["watching_status"].isin([2, 4])].groupby("anime_id").size()
candidate_universe = registrations.sort_values(ascending=False).head(3000).index.tolist()
anime_avg_lookup = anime_avg["anime_avg_completion_rate_A"].to_dict()
avg_mean_fallback = anime_avg["anime_avg_completion_rate_A"].mean()

N_CANDIDATES = 200
overlaps = []
n_evaluated = 0

for uid in test_users:
    if uid not in labels_by_user:
        continue
    g = labels_by_user[uid]
    rng = np.random.default_rng(SEED + int(uid))
    rows = g[["anime_id", "watching_status"]].values.tolist()
    pool_rows = [r for r in rows if r[0] in pool_ids]
    perm_idx = rng.permutation(len(pool_rows))
    input_rows = [pool_rows[i] for i in perm_idx][:20]
    if len(input_rows) == 0:
        continue
    responses = [make_response(aid, st) for aid, st in input_rows]
    endurance = compute_endurance_episodes(responses)
    completion_rate = compute_completion_rate(responses)
    comp_vec, drop_vec = profile_vectors(responses)
    comp_pop = [popularity_lookup.get(r.anime_id, np.nan) for r in responses if r.label == "completed"]
    user_mean_pop = float(np.mean(comp_pop)) if comp_pop else np.nan
    genre_rate = {}
    for g_name in all_genres:
        c = sum(1 for r in responses if r.label == "completed" and g_name in r.genres)
        d = sum(1 for r in responses if r.label == "dropped" and g_name in r.genres)
        if c + d > 0:
            genre_rate[g_name] = c / (c + d)

    answered_ids = set(labels_all[labels_all["user_id"] == uid]["anime_id"])
    unrated = [aid for aid in candidate_universe if aid not in answered_ids]
    if len(unrated) < N_CANDIDATES:
        continue
    candidates = rng.choice(unrated, size=N_CANDIDATES, replace=False)

    cand_rows = []
    for aid in candidates:
        a_genres = genres_lookup.get(aid) or []
        a_episodes = episodes_lookup.get(aid)
        a_emb = emb_lookup.get(aid)
        cos_c = cos_sim(a_emb, comp_vec); cos_d = cos_sim(a_emb, drop_vec)
        ep_gap = (a_episodes - endurance) if (a_episodes is not None and endurance is not None) else np.nan
        gmatch = [genre_rate[g_name] for g_name in a_genres if g_name in genre_rate]
        genre_match_rate = float(np.mean(gmatch)) if gmatch else np.nan
        a_pop = popularity_lookup.get(aid, np.nan)
        pop_div = (a_pop - user_mean_pop) if not np.isnan(user_mean_pop) else np.nan
        row = {"anime_id": aid, "endurance_episodes": endurance, "user_completion_rate": completion_rate,
               "cos_completed": cos_c, "cos_dropped": cos_d, "episode_gap": ep_gap,
               "genre_match_completion_rate": genre_match_rate, "popularity_divergence": pop_div,
               "anime_avg_completion_rate_A": anime_avg_lookup.get(aid, avg_mean_fallback),
               "popularity_percentile": a_pop}
        for sc in STATIC_COLS:
            if sc not in row:
                row[sc] = anime_feat.loc[aid, sc] if aid in anime_feat.index and sc in anime_feat.columns else np.nan
        cand_rows.append(row)

    cand_df = pd.DataFrame(cand_rows)
    cand_df["source"] = cand_df["source"].astype("category")
    for c in feat3:
        if c not in cand_df.columns:
            cand_df[c] = np.nan
    cand_df["pred_dropped"] = model_full.predict(cand_df[feat3])
    cand_df["pred_completion"] = 1 - cand_df["pred_dropped"]

    top_pop = set(cand_df.nlargest(20, "popularity_percentile")["anime_id"])
    top_personal = set(cand_df.nlargest(20, "pred_completion")["anime_id"])
    overlaps.append(len(top_pop & top_personal))
    n_evaluated += 1

overlaps = np.array(overlaps)
print(f"評価ユーザー数: {n_evaluated:,}")
print(f"重複件数: 平均={overlaps.mean():.2f} 中央値={np.median(overlaps):.1f} / 20")


評価ユーザー数: 2,158
重複件数: 平均=0.86 中央値=1.0 / 20


## 11. 追加分析④: 理論上限の見積もりと達成率の議論

In [15]:
n_completed_all = (labels["watching_status"]==2).sum()
n_dropped_A_all = (labels["watching_status"]==4).sum()
n_onhold_all = (labels_all["watching_status"]==3).sum()
print(f"完走: {n_completed_all:,} / 離脱(A): {n_dropped_A_all:,} / 保留(学習には不使用): {n_onhold_all:,}")
print(f"\n参考: N=30モデルのAUC = {n_curve_df[n_curve_df.N==30]['auc'].values[0]:.4f}")
print(f"N=20→30の伸び(AUC): {n_curve_df[n_curve_df.N==30]['auc'].values[0] - n_curve_df[n_curve_df.N==20]['auc'].values[0]:.4f}")
print(f"N=20→30の伸び(PR-AUC): {n_curve_df[n_curve_df.N==30]['pr_auc'].values[0] - n_curve_df[n_curve_df.N==20]['pr_auc'].values[0]:.4f}")


完走: 3,138,037 / 離脱(A): 202,469 / 保留(学習には不使用): 170,045

参考: N=30モデルのAUC = 0.8409
N=20→30の伸び(AUC): 0.0030
N=20→30の伸び(PR-AUC): 0.0062


## 12. 埋め込みの新作パス検証（コールドスタート）【新規】

`anime_avg_completion_rate` を除外した特徴量セットで、埋め込みあり/なしを比較する。
これは2021年以降の新作（過去実績なし、`is_estimated: true`）の判定精度に直結する。

In [16]:
base_feat = USER_COLS + STATIC_COLS  # anime_avg抜き

m_no_emb = train_lgbm(base_feat, tr20, va20, CAT_COLS)
pred_no = m_no_emb.predict(te20[base_feat], num_iteration=m_no_emb.best_iteration)
auc_no, pr_no = auc_pr(te20["label_dropped"], pred_no)

emb_feat = base_feat + ["cos_completed", "cos_dropped"]
m_emb = train_lgbm(emb_feat, tr20, va20, CAT_COLS)
pred_yes = m_emb.predict(te20[emb_feat], num_iteration=m_emb.best_iteration)
auc_yes, pr_yes = auc_pr(te20["label_dropped"], pred_yes)

full_auc = [r for r in results_main if r["stage"] == "+離脱ベクトル"][0]["auc"]
print(f"anime_avg除外・埋め込みなし: AUC={auc_no:.4f} PR-AUC={pr_no:.4f}")
print(f"anime_avg除外・埋め込みあり: AUC={auc_yes:.4f} PR-AUC={pr_yes:.4f}")
print(f"\n参考: anime_avgありのフルモデル AUC = {full_auc:.4f}")
print(f"埋め込みによる回復幅: {auc_yes - auc_no:.4f}")
print(f"anime_avg除外による損失（埋め込みなし基準）: {full_auc - auc_no:.4f}")


anime_avg除外・埋め込みなし: AUC=0.8353 PR-AUC=0.3183
anime_avg除外・埋め込みあり: AUC=0.8351 PR-AUC=0.3184

参考: anime_avgありのフルモデル AUC = 0.8379
埋め込みによる回復幅: -0.0002
anime_avg除外による損失（埋め込みなし基準）: 0.0025


---

## まとめ

全ての評価表・追加分析の結果は `reports/model_evaluation.md` にまとめる。
Phase 3（API実装）にはまだ着手していない。